# Event studies: replication

Energy Research Warehouse (ERW), session 47. This notebook reproduces every event study estimate shown on the `/events` pages (and every daily and pooled estimate of the table `event_study_estimates`) from `event_window_daily`, fetched with the `erw` package. The method is `docs/methods/event_study.md`; this is a third implementation of it, written from the method, in numpy.

Run it from the repository root (or set `ERW_DATA_DIR` to a folder of ERW tables). No network beyond the package's own backend, no model.

In [ ]:
import numpy as np
import pandas as pd
import erw

Z = 1.959963984540054
WINDOWS = {  # local dates, inclusive (warehouse/derived/event_window.py)
    "uri_2021": ("2021-02-07", "2021-02-24"),
    "covid_2020": ("2020-03-01", "2020-05-31"),
    "caiso_heat_2020": ("2020-08-10", "2020-08-24"),
    "elliott_2022": ("2022-12-19", "2022-12-29"),
    "ercot_heat_2023": ("2023-08-01", "2023-09-10"),
    "caiso_heat_2022": ("2022-08-31", "2022-09-09"),  # session 58
}
win = erw.fetch("event_window_daily")
win = win[(win["freq"] == "P1D") & win["variable"].isin(["demand_mwh", "rt_mean"])].copy()
win["day"] = win["ts_utc"].dt.strftime("%Y-%m-%d")
len(win)

## The estimator

The regression: an intercept, six weekday indicators (Monday the reference), year contrasts summing to zero over the baseline years (the event year takes none), and either one indicator per event day or one pooled indicator. Each day's effect is its value less the baseline-only fit at that day, with standard error sqrt(s2 + x'Vx); the pooled effect's standard error is HC1.

In [ ]:
def columns(days, event, base_years, pooled, Z=None):
    d = pd.to_datetime(pd.Series(days))
    X = {"intercept": np.ones(len(d))}
    if pooled:
        X["pooled"] = np.asarray(event, dtype=float)
    for k in range(1, 7):
        X[f"dow{k}"] = (d.dt.dayofweek == k).to_numpy(dtype=float)
    if len(base_years) >= 2:
        for y in base_years[:-1]:
            X[f"year{y}"] = np.where(event, 0.0, (d.dt.year == y).astype(float) - (d.dt.year == base_years[-1]).astype(float))
    if Z is not None:  # session 49: the grid's degree days and their squares
        for j in range(Z.shape[1]):
            X[f"z{j}"] = Z[:, j]
    return pd.DataFrame(X)


def independent(X):
    """The columns kept: not all zero, and not a linear combination of the columns kept before them (greedy Gram-Schmidt,
    relative tolerance 1e-9), as warehouse/derived/event_study.py drops them."""
    kept, basis = [], []
    for c in X.columns:
        v = X[c].to_numpy(float)
        n = np.linalg.norm(v)
        if n == 0:
            continue
        r = v.copy()
        for q in basis:
            r = r - (q @ r) * q
        if np.linalg.norm(r) > 1e-9 * n:
            kept.append(c)
            basis.append(r / np.linalg.norm(r))
    return X[kept]


def hc1(X, y):
    X, y = np.asarray(X, float), np.asarray(y, float)
    n, k = X.shape
    A = np.linalg.inv(X.T @ X)
    b = A @ X.T @ y
    e = y - X @ b
    V = A @ (X.T * e ** 2) @ X @ A * n / (n - k)
    return b, V, e @ e / (n - k)


def study(g, event_id, weather=None):
    g = g.sort_values("day")
    Z = None
    if weather is not None:  # session 49: the days with weather only, and hdd, hdd^2, cdd, cdd^2
        g = g[g["day"].isin(weather.index)]
        w = weather.loc[g["day"]]
        Z = np.column_stack([w["hdd"], w["hdd"] ** 2, w["cdd"], w["cdd"] ** 2])
    s, t = WINDOWS[event_id]
    ev = ((g["day"] >= s) & (g["day"] <= t)).to_numpy()
    y = g["value"].to_numpy(float)
    base_years = sorted(set(pd.to_datetime(g["day"][~ev]).dt.year))
    X = columns(g["day"], ev, base_years, False, Z)
    Xb = independent(X[~ev])
    b, V, s2 = hc1(Xb, y[~ev])
    Xe = X[ev][Xb.columns].to_numpy()
    cf = Xe @ b
    days = pd.DataFrame({"day": g["day"][ev].to_numpy(), "estimate": y[ev] - cf,
                         "se": np.sqrt(s2 + np.einsum("ij,jk,ik->i", Xe, V, Xe)), "counterfactual": cf})
    P = columns(g["day"], ev, base_years, True, Z)
    P = independent(P)
    bp, Vp, _ = hc1(P, y)
    j = list(P.columns).index("pooled")
    return days, bp[j], np.sqrt(Vp[j, j]), cf.mean()

## Every estimate

One row per event, grid and outcome: the pooled effect, its 95 percent interval, the mean counterfactual, and the effect as a percent of it.

In [ ]:
pooled, daily = [], []
for (event_id, entity, variable), g in win.groupby(["event", "entity", "variable"]):
    days, est, se, cf = study(g, event_id)
    pooled.append(dict(event=event_id, entity=entity, variable=variable, estimate=est, se=se,
                       ci_low=est - Z * se, ci_high=est + Z * se, counterfactual_mean=cf, pct=est / cf * 100, n=len(g)))
    daily.append(days.assign(event=event_id, entity=entity, variable=variable))
pooled = pd.DataFrame(pooled)
daily = pd.concat(daily, ignore_index=True)
pooled.round(2)

## The same as the warehouse's table

`event_study_estimates` holds the estimates of `warehouse/derived/event_study.py`; the pages compute theirs with `site/lib/eventstudy.ts`, held equal to that table by `site/scripts/test-eventstudy.mjs`. Every daily and pooled estimate and standard error here equals the table's.

In [ ]:
tab = erw.fetch("event_study_estimates")
tab["day"] = tab["ts_utc"].dt.strftime("%Y-%m-%d")
tab["x_std_error"] = pd.to_numeric(tab["x_std_error"])  # the x_ columns come back as text
p = tab[tab["variable"].str.endswith("_effect_pooled")].assign(variable=lambda t: t["variable"].str.replace("_effect_pooled", ""))
m = pooled.merge(p, on=["event", "entity", "variable"])
d = tab[tab["variable"].str.endswith("_effect_day")].assign(variable=lambda t: t["variable"].str.replace("_effect_day", ""))
k = daily.merge(d, on=["event", "entity", "variable", "day"])
rel = lambda a, b: ((a - b).abs() / b.abs().clip(lower=1)).max()  # relative to the value (floating point)
worst = max(rel(m["estimate"], m["value"]), rel(m["se"], m["x_std_error"]), rel(k["estimate"], k["value"]), rel(k["se"], k["x_std_error"]))
assert len(m) == len(pooled) and len(k) == len(daily) and worst < 1e-6, worst
print(f"{len(m)} pooled and {len(k)} daily estimates equal the table's (largest relative difference {worst:.2e})")

## Controlling for temperature (session 49)

The second specification adds, for each day, the grid's heating and cooling degree days at 65 F and their squares: the mean over the grid's weather stations of `event_window_daily`'s station rows (NOAA ISD), as `warehouse/derived/event_study.py` builds them. Days without weather are left out of this specification only. Every pooled and daily temperature-controlled estimate equals the table's.

In [ ]:
raw = erw.fetch("event_window_daily")
st = raw[raw["entity"].str.startswith("noaa:") & raw["variable"].isin(["hdd_65f", "cdd_65f"])].copy()
st["day"] = st["ts_utc"].dt.strftime("%Y-%m-%d")
grid_of = {"eia930:CISO": "ciso", "eia930:ERCO": "erco", "eia930:ISNE": "isne", "eia930:MISO": "miso", "eia930:NYIS": "nyis",
           "eia930:PJM": "pjm", "eia930:SWPP": "swpp", "ercot:HB_HUBAVG": "erco"}
pooled_t, daily_t = [], []
for (event_id, entity, variable), g in win.groupby(["event", "entity", "variable"]):
    ba = grid_of.get(entity)
    w = st[(st["event"] == event_id) & (st["ba"] == ba)] if ba else st.iloc[0:0]
    if not len(w):
        continue
    wd = w.pivot_table(index="day", columns="variable", values="value", aggfunc="mean").rename(columns={"hdd_65f": "hdd", "cdd_65f": "cdd"})[["hdd", "cdd"]].dropna()
    days, est, se, cf = study(g, event_id, wd)
    pooled_t.append(dict(event=event_id, entity=entity, variable=variable, estimate=est, se=se))
    daily_t.append(days.assign(event=event_id, entity=entity, variable=variable))
pooled_t = pd.DataFrame(pooled_t)
daily_t = pd.concat(daily_t, ignore_index=True)
pt = tab[tab["variable"].str.endswith("_effect_pooled_temp")].assign(variable=lambda t: t["variable"].str.replace("_effect_pooled_temp", ""))
dt_ = tab[tab["variable"].str.endswith("_effect_day_temp")].assign(variable=lambda t: t["variable"].str.replace("_effect_day_temp", ""))
mt = pooled_t.merge(pt, on=["event", "entity", "variable"])
kt = daily_t.merge(dt_, on=["event", "entity", "variable", "day"])
worst_t = max(rel(mt["estimate"], mt["value"]), rel(mt["se"], mt["x_std_error"]), rel(kt["estimate"], kt["value"]), rel(kt["se"], kt["x_std_error"]))
assert len(mt) == len(pooled_t) == len(pt) and len(kt) == len(daily_t) and worst_t < 1e-6, (len(mt), len(pooled_t), len(pt), worst_t)
print(f"{len(mt)} pooled and {len(kt)} daily temperature-controlled estimates equal the table's (largest relative difference {worst_t:.2e})")
pooled_t.merge(pooled, on=["event", "entity", "variable"], suffixes=("_temp", "")).round(2)[["event", "entity", "variable", "estimate", "estimate_temp"]]